# 01. Genuine Sentinel-2 Data Acquisition Pipeline
**Project:** Attention U-Net Based Glacier Segmentation Using Sentinel-2 and Statistical Comparison with Landsat Satellite Data  
**Study Area:** Gangotri Glacier Basin, Uttarakhand, India (Garhwal Himalaya)  
**Primary Satellite Sensor:** Copernicus Sentinel-2 MSI Level-2A (Surface Reflectance)  

### Scientific Objectives:
1. Query AWS Earth Search STAC for cloud-free (<5%) Sentinel-2 Level-2A scenes over MGRS Tile `44RLV`.
2. Extract multi-temporal observations across post-monsoon ablation seasons (2021, 2022, 2023, 2024).
3. Establish a common 30m grid aligned with NASA Harmonized Landsat-Sentinel (HLS) standards.
4. Stream spectral bands directly via Cloud-Optimized GeoTIFFs (COGs).


In [ ]:
# Google Colab Setup (Optional: Mount Google Drive for persistent storage)
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/glacier_data'
    print("[+] Mounted Google Drive successfully.")
except ImportError:
    DATA_ROOT = '../data'
    print("[*] Running in local/scratch environment.")

os.makedirs(DATA_ROOT, exist_ok=True)


In [ ]:
import requests
import json
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

# Gangotri Basin Geographic Bounding Box
BBOX_WGS84 = [78.95, 30.70, 79.45, 31.15]
STAC_URL = "https://earth-search.aws.element84.com/v1/search"

# Query STAC API for cloud-free ablation season scenes
payload = {
    "collections": ["sentinel-2-l2a"],
    "bbox": BBOX_WGS84,
    "datetime": "2021-09-01T00:00:00Z/2024-11-20T23:59:59Z",
    "query": {"eo:cloud_cover": {"lt": 5.0}},
    "limit": 20,
    "sortby": [{"field": "properties.eo:cloud_cover", "direction": "asc"}]
}

resp = requests.post(STAC_URL, json=payload, timeout=20)
scenes = resp.json().get("features", [])
print(f"[+] Found {len(scenes)} cloud-free Sentinel-2 scenes:")
for s in scenes[:5]:
    dt = s["properties"]["datetime"]
    cc = s["properties"]["eo:cloud_cover"]
    print(f"  • Scene: {s['id']} | Date: {dt} | Cloud: {cc:.2f}%")


In [ ]:
# Inspect spectral assets for target scene
selected_scene = scenes[0]
print(f"Target Scene: {selected_scene['id']}")
print(f"Available Spectral Assets:")
for band_name, asset in selected_scene["assets"].items():
    if "image" in asset.get("type", ""):
        print(f"  - {band_name:10s}: {asset['href'][:80]}...")
